## Cell 1 — Install Dependencies

In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 beautifulsoup4 transformers torch sentence-transformers faiss-cpu accelerate spacy networkx pyvis bitsandbytes wordcloud matplotlib Pillow -q
!python -m spacy download en_core_web_sm -q
print('✅ All dependencies installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 117.5 MB/s eta 0:00:00
✔ Download and installation successful
You can no

## Cell 2 — Mount Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

APP_DIR = '/content/drive/MyDrive/PolicyNav'
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'documents'), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'graphs'), exist_ok=True)
os.environ['APP_DIR'] = APP_DIR

print(f'✅ APP_DIR set to: {APP_DIR}')


Mounted at /content/drive
✅ APP_DIR set to: /content/drive/MyDrive/PolicyNav


## Cell 3 — `config.py`

In [ ]:
%%writefile config.py
import os

JWT_SECRET             = os.getenv('JWT_SECRET_KEY')
JWT_ALGORITHM          = 'HS256'
TOKEN_EXPIRY_MINUTES   = 60

MAX_LOGIN_ATTEMPTS     = 3
LOCK_TIME_MINUTES      = 5
PASSWORD_HISTORY_COUNT = 3

EMAIL_ID               = os.getenv('EMAIL_ID')
EMAIL_APP_PASSWORD     = os.getenv('EMAIL_APP_PASSWORD')

ADMIN_EMAIL            = os.getenv('ADMIN_EMAIL_ID')
ADMIN_PASSWORD         = os.getenv('ADMIN_PASSWORD')


Writing config.py


In [ ]:
%%writefile db.py
# ============================================================
# db.py — PolicyNav Database (Milestone 4 updated)
# ============================================================
import os
import sqlite3
import json
from datetime import datetime

DB_NAME = os.path.join(os.environ.get('APP_DIR', '.'), 'policynav_users.db')

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def _safe_add_column(cursor, table, column, col_type):
    """Adds a column only if it doesn't already exist — safe migration."""
    try:
        cursor.execute(f'ALTER TABLE {table} ADD COLUMN {column} {col_type}')
    except Exception:
        pass

def init_db():
    conn   = get_connection()
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id                   INTEGER PRIMARY KEY AUTOINCREMENT,
        username             TEXT UNIQUE,
        email                TEXT UNIQUE,
        password_hash        TEXT,
        security_question    TEXT,
        security_answer_hash TEXT,
        failed_attempts      INTEGER DEFAULT 0,
        lock_until           TEXT,
        password_history     TEXT,
        is_admin             INTEGER DEFAULT 0
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS activity_log (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        user_email TEXT,
        app_section TEXT,
        user_input  TEXT,
        ai_output   TEXT,
        created_at  TEXT DEFAULT CURRENT_TIMESTAMP
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS feedback (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        user_email TEXT,
        section    TEXT,
        rating     INTEGER,
        comments   TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
    ''')

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS user_settings (
        email            TEXT PRIMARY KEY,
        app_language     TEXT    DEFAULT 'English',
        notifications    INTEGER DEFAULT 1,
        show_timestamps  INTEGER DEFAULT 1,
        results_per_page INTEGER DEFAULT 5,
        updated_at       TEXT    DEFAULT CURRENT_TIMESTAMP
    )
    ''')

    conn.commit()

    # ── Safe migrations (won't break existing DB) ──
    _safe_add_column(cursor, 'users', 'is_admin',           'INTEGER DEFAULT 0')
    _safe_add_column(cursor, 'user_settings', 'notifications',    'INTEGER DEFAULT 1')
    _safe_add_column(cursor, 'user_settings', 'show_timestamps',  'INTEGER DEFAULT 1')
    _safe_add_column(cursor, 'user_settings', 'results_per_page', 'INTEGER DEFAULT 5')
    _safe_add_column(cursor, 'users', 'avatar',             'TEXT DEFAULT NULL')
    _safe_add_column(cursor, 'users', 'new_email_pending',  'TEXT DEFAULT NULL')
    _safe_add_column(cursor, 'activity_log', 'language',    'TEXT DEFAULT "English"')
    conn.commit()
    conn.close()

# ── Original teammate functions (unchanged) ──────────────────
def create_user(username, email, password_hash, question, answer_hash):
    conn   = get_connection()
    cursor = conn.cursor()
    try:
        history = json.dumps([password_hash])
        cursor.execute('''
        INSERT INTO users (username, email, password_hash, security_question,
                           security_answer_hash, password_history)
        VALUES (?, ?, ?, ?, ?, ?)
        ''', (username, email, password_hash, question, answer_hash, history))
        conn.commit()
        return True, 'User created'
    except:
        return False, 'User exists'
    finally:
        conn.close()

def get_user_by_email(email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM users WHERE email=?', (email,))
    user   = cursor.fetchone()
    conn.close()
    return user

def update_login_attempts(email, attempts, lock_until=None):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?',
        (attempts, lock_until, email)
    )
    conn.commit()
    conn.close()

def update_password(email, new_hash):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('UPDATE users SET password_hash=? WHERE email=?', (new_hash, email))
    conn.commit()
    conn.close()

def update_password_history(email, history_json):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'UPDATE users SET password_history=? WHERE email=?',
        (history_json, email)
    )
    conn.commit()
    conn.close()

def get_all_users():
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, username, email, failed_attempts, lock_until, is_admin FROM users')
    rows   = cursor.fetchall()
    conn.close()
    return rows

def promote_user(email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('UPDATE users SET is_admin=1 WHERE email=?', (email,))
    conn.commit()
    conn.close()

def demote_user(email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('UPDATE users SET is_admin=0 WHERE email=?', (email,))
    conn.commit()
    conn.close()

def delete_user(email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('DELETE FROM users WHERE email=?', (email,))
    deleted_label = f'[DELETED] {email}'
    cursor.execute('UPDATE activity_log SET user_email=? WHERE user_email=?', (deleted_label, email))
    cursor.execute('UPDATE feedback SET user_email=? WHERE user_email=?',     (deleted_label, email))
    conn.commit()
    conn.close()

def get_all_activity_logs():
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, user_email, app_section, user_input, ai_output, created_at FROM activity_log ORDER BY created_at DESC')
    rows   = cursor.fetchall()
    conn.close()
    return rows

def submit_feedback(user_email, section, rating, comments):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'INSERT INTO feedback (user_email, section, rating, comments) VALUES (?, ?, ?, ?)',
        (user_email, section, rating, comments)
    )
    conn.commit()
    conn.close()

def log_activity(user_email, section, user_input, ai_output, language="English"):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'INSERT INTO activity_log (user_email, app_section, user_input, ai_output, language) VALUES (?, ?, ?, ?, ?)',
        (user_email, section, user_input, ai_output, language)
    )
    conn.commit()
    conn.close()

def get_user_activity(user_email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'SELECT app_section, user_input, ai_output, created_at FROM activity_log WHERE user_email=? ORDER BY created_at DESC',
        (user_email,)
    )
    rows   = cursor.fetchall()
    conn.close()
    return rows

def get_all_feedback():
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'SELECT user_email, section, rating, comments, created_at FROM feedback ORDER BY created_at DESC'
    )
    rows   = cursor.fetchall()
    conn.close()
    return rows

def lock_user(email):
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'UPDATE users SET failed_attempts=999, lock_until=? WHERE email=?',
        ('9999-12-31T23:59:59', email)
    )
    conn.commit()
    conn.close()

def get_language_counts():
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT language, COUNT(*) as cnt FROM activity_log WHERE language IS NOT NULL GROUP BY language ORDER BY cnt DESC"
    )
    rows = cursor.fetchall()
    conn.close()
    return rows

# ── Milestone 4 additions ────────────────────────────────────

def get_avatar(email):
    """Returns Base64 avatar string or None."""
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT avatar FROM users WHERE email=?', (email,))
    row    = cursor.fetchone()
    conn.close()
    return row[0] if row and row[0] else None

def save_avatar(email, base64_string):
    """Saves or clears the avatar for a user."""
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('UPDATE users SET avatar=? WHERE email=?', (base64_string, email))
    conn.commit()
    conn.close()

def store_pending_email(current_email, new_email):
    """Temporarily stores the new email before OTP verification."""
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'UPDATE users SET new_email_pending=? WHERE email=?',
        (new_email, current_email)
    )
    conn.commit()
    conn.close()

def confirm_email_update(current_email):
    """
    After OTP verified: moves new_email_pending → email.
    Updates all related tables. Returns new email or None.
    """
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT new_email_pending FROM users WHERE email=?', (current_email,))
    row    = cursor.fetchone()
    if not row or not row[0]:
        conn.close()
        return None
    new_email = row[0]
    try:
        cursor.execute(
            'UPDATE users SET email=?, new_email_pending=NULL WHERE email=?',
            (new_email, current_email)
        )
        cursor.execute('UPDATE activity_log SET user_email=? WHERE user_email=?', (new_email, current_email))
        cursor.execute('UPDATE feedback SET user_email=? WHERE user_email=?',     (new_email, current_email))
        conn.commit()
        return new_email
    except Exception:
        conn.rollback()
        return None
    finally:
        conn.close()

def get_user_profile(email):
    """
    Returns profile summary dict for the Profile page.
    { email, username, created_at, avatar,
      total_activities, total_feedback, last_active }
    """
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT username, avatar FROM users WHERE email=?', (email,))
    user_row = cursor.fetchone()
    cursor.execute('SELECT COUNT(*) FROM activity_log WHERE user_email=?', (email,))
    total_activities = cursor.fetchone()[0]
    cursor.execute('SELECT COUNT(*) FROM feedback WHERE user_email=?', (email,))
    total_feedback = cursor.fetchone()[0]
    cursor.execute(
        'SELECT created_at FROM activity_log WHERE user_email=? ORDER BY created_at DESC LIMIT 1',
        (email,)
    )
    last_row    = cursor.fetchone()
    last_active = last_row[0] if last_row else 'No activity yet'
    conn.close()
    if not user_row:
        return None
    return {
        'email':            email,
        'username':         user_row[0] or email.split('@')[0],
        'avatar':           user_row[1],
        'total_activities': total_activities,
        'total_feedback':   total_feedback,
        'last_active':      last_active,
    }

def get_user_settings(email):
    """Returns all user settings as a dict. Returns defaults if not set."""
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        'SELECT app_language, notifications, show_timestamps, results_per_page '
        'FROM user_settings WHERE email=?', (email,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            'app_language':     row[0] or 'English',
            'notifications':    bool(row[1]) if row[1] is not None else True,
            'show_timestamps':  bool(row[2]) if row[2] is not None else True,
            'results_per_page': row[3] or 5,
        }
    return {
        'app_language':     'English',
        'notifications':    True,
        'show_timestamps':  True,
        'results_per_page': 5,
    }

def save_user_settings(email, app_language,
                       notifications=True,
                       show_timestamps=True,
                       results_per_page=5):
    """Saves all user settings. Creates row if not exists (upsert)."""
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO user_settings
            (email, app_language, notifications,
             show_timestamps, results_per_page, updated_at)
        VALUES (?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
        ON CONFLICT(email) DO UPDATE SET
            app_language     = excluded.app_language,
            notifications    = excluded.notifications,
            show_timestamps  = excluded.show_timestamps,
            results_per_page = excluded.results_per_page,
            updated_at       = CURRENT_TIMESTAMP
    """, (email, app_language,
             int(notifications), int(show_timestamps),
             results_per_page))
    conn.commit()
    conn.close()



Writing db.py


## Cell 5 — `auth.py` (Unchanged)

In [ ]:
%%writefile auth.py
import re
import bcrypt
import jwt
import random
import smtplib
from datetime import datetime, timedelta
from email.mime.text import MIMEText
from config import *

def hash_text(text: str) -> str:
    return bcrypt.hashpw(text.encode(), bcrypt.gensalt()).decode()

def verify_text(text: str, hashed: str) -> bool:
    return bcrypt.checkpw(text.encode(), hashed.encode())

def validate_email(email: str) -> bool:
    return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email) is not None

def validate_password(password: str):
    if len(password) < 8:                      return False, 'Password must be at least 8 characters'
    if not re.search(r'[A-Z]', password):      return False, 'Password must contain at least 1 uppercase letter'
    if not re.search(r'[a-z]', password):      return False, 'Password must contain at least 1 lowercase letter'
    if not re.search(r'\d', password):         return False, 'Password must contain at least 1 number'
    return True, 'Valid password'

def generate_token(email: str):
    payload = {
        'email': email,
        'exp':   datetime.utcnow() + timedelta(minutes=TOKEN_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

def verify_token(token: str):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except:
        return None

def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp_email(receiver, otp):
    html_content = f"""
    <html><body style='font-family:Arial,sans-serif;background:#f4f4f4;padding:20px;'>
        <div style='max-width:400px;margin:auto;background:white;padding:20px;
                    border-radius:10px;text-align:center;
                    box-shadow:0 4px 10px rgba(0,0,0,0.1);'>
            <h2 style='color:#2c3e50;'>PolicyNav Verification</h2>
            <p style='color:#555;'>Use the OTP below to continue:</p>
            <div style='font-size:28px;font-weight:bold;letter-spacing:3px;
                        color:#00b894;margin:20px 0;'>{otp}</div>
            <p style='color:#999;font-size:12px;'>This OTP is valid for 10 minutes.</p>
        </div>
    </body></html>
    """
    msg            = MIMEText(html_content, 'html')
    msg['Subject'] = 'PolicyNav OTP Verification'
    msg['From']    = EMAIL_ID
    msg['To']      = receiver
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(EMAIL_ID, EMAIL_APP_PASSWORD)
        server.send_message(msg)


Writing auth.py


## Cell 6 — `readability.py` (Unchanged)

In [ ]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text

    def get_all_metrics(self):
        metrics = {
            'Flesch Reading Ease':  textstat.flesch_reading_ease(self.text),
            'Flesch-Kincaid Grade': textstat.flesch_kincaid_grade(self.text),
            'SMOG Index':           textstat.smog_index(self.text),
            'Gunning Fog':          textstat.gunning_fog(self.text),
            'Coleman-Liau':         textstat.coleman_liau_index(self.text),
        }
        metrics['Overall Grade Level'] = (
            metrics['Flesch-Kincaid Grade'] + metrics['SMOG Index'] +
            metrics['Gunning Fog'] + metrics['Coleman-Liau']
        ) / 4
        metrics['Total Sentences']  = textstat.sentence_count(self.text)
        metrics['Total Words']      = textstat.lexicon_count(self.text)
        metrics['Total Syllables']  = textstat.syllable_count(self.text)
        metrics['Complex Words']    = textstat.difficult_words(self.text)
        metrics['Total Characters'] = len(self.text)
        return metrics


Writing readability.py


## Cell 7 — `vector_store.py` (Unchanged)

In [ ]:
%%writefile vector_store.py
import os, glob, pickle, faiss, PyPDF2
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

APP_DIR = os.environ.get('APP_DIR', '.')
INDEX_PATH = os.path.join(APP_DIR, "faiss_index.bin")
META_PATH = os.path.join(APP_DIR, "faiss_meta.pkl")

# We only load it once cached in Streamlit
_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _embedder

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        for page in PyPDF2.PdfReader(pdf_path).pages: text += page.extract_text() + "\n"
    except: pass
    return text

def ingest_documents(docs_dir):
    if not os.path.exists(docs_dir): return 0

    # Load existing metdata to avoid re-ingesting
    existing_metadata = []
    if os.path.exists(META_PATH):
        try:
            with open(META_PATH, 'rb') as f: existing_metadata = pickle.load(f)
        except: pass

    existing_filenames = set([d['filename'] for d in existing_metadata])

    files = glob.glob(os.path.join(docs_dir, "*"))
    new_chunks = []
    new_metadata = []

    for filepath in files:
        filename = os.path.basename(filepath)
        if filename in existing_filenames: continue # Skip already ingested!

        print(f"Parsing new document: {filename}")
        text = ""
        if filepath.lower().endswith(".pdf"): text = extract_text_from_pdf(filepath)
        elif filepath.lower().endswith((".htm", ".html")): text = BeautifulSoup(open(filepath, 'r').read(), 'html.parser').get_text(separator=' ')
        elif filepath.lower().endswith(".txt"): text = open(filepath, 'r').read()

        if text.strip():
            for i in range(0, len(text), 1500):
                chunk = text[i:i+1500]
                if len(chunk) > 50:
                    new_chunks.append(chunk)
                    new_metadata.append({"filename": filename, "content": chunk})

    if not new_chunks: return 0 # Nothing new to ingest

    embedder = get_embedder()
    embeddings = embedder.encode(new_chunks, convert_to_numpy=True)

    if os.path.exists(INDEX_PATH):
        index = faiss.read_index(INDEX_PATH)
    else:
        index = faiss.IndexFlatL2(embeddings.shape[1])

    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

    final_metadata = existing_metadata + new_metadata
    with open(META_PATH, 'wb') as f: pickle.dump(final_metadata, f)
    return len(new_chunks)

def search_documents(query, top_k=5):
    if not os.path.exists(INDEX_PATH) or not os.path.exists(META_PATH): return []
    try:
        embedder = get_embedder()
        index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, 'rb') as f: metadata = pickle.load(f)
        distances, indices = index.search(embedder.encode([query], convert_to_numpy=True), top_k)

        # Deduplicate to try to get broader file sources rather than just 5 chunks from the exact same page
        results = []
        seen_files = set()
        for idx in indices[0]:
            if idx != -1 and idx < len(metadata):
                doc = metadata[idx]
                fname = doc['filename']
                if list(seen_files).count(fname) < 2: # Max 2 chunks from same file allowed
                    results.append(doc)
                    seen_files.add(fname)
        return results
    except: return []

def get_all_documents():
    if not os.path.exists(META_PATH): return []
    with open(META_PATH, 'rb') as f: return pickle.load(f)


Writing vector_store.py


## Cell 8 — `knowledge_graph.py` (Unchanged)

In [ ]:
%%writefile knowledge_graph.py
import spacy, networkx as nx, os
from pyvis.network import Network

try: nlp = spacy.load("en_core_web_sm")
except: nlp = None

APP_DIR = os.environ.get('APP_DIR', '.')

def build_graph_from_documents(docs):
    G = nx.Graph()
    max_docs = min(len(docs), 50)
    if not nlp: return G

    for i in range(max_docs):
        text = docs[i]['content'][:1000]
        doc = nlp(text)
        entities = [
            (ent.text.strip(), ent.label_) for ent in doc.ents
            if ent.label_ in ['ORG','GPE','LAW','PERSON'] and len(ent.text.strip()) > 2
        ]

        entities = list(dict.fromkeys(entities))[:15]

        source_node = f"Doc: {docs[i]['filename']}"
        G.add_node(
            source_node,
            label=source_node,
            title=f"📄 Document: {docs[i]['filename']}",
            color="#38bdf8",
            size=18
        )
        for ent, label in entities:

            if label == "ORG":
                color = "#22c55e"   # green
            elif label == "PERSON":
                color = "#f472b6"   # pink
            elif label == "LAW":
                color = "#f59e0b"   # orange
            elif label == "GPE":
                color = "#eab308"   # yellow
            else:
                color = "#22c55e"

            G.add_node(
                ent,
                label=ent,
                title=f"""
                Entity: {ent}
                Type: {label}
                Source Document: {docs[i]['filename']}
                """,
                color=color,
                size=20
            )
            G.add_edge(source_node, ent)

        # NEW: connect entities with each other
        entity_names = [e[0] for e in entities]

        for i in range(min(len(entity_names), 5)):
            for j in range(i + 1, min(len(entity_names), 5)):
                G.add_edge(entity_names[i], entity_names[j])
    return G

def generate_interactive_graph(docs):
    if not docs: return None
    G = build_graph_from_documents(docs)

    net = Network(
        height="650px",
        width="100%",
        bgcolor="#0b0f19",
        font_color="white",
        notebook=False
    )

    net.barnes_hut(
        gravity=-8000,
        central_gravity=0.3,
        spring_length=200
    )

    for node in G.nodes():
        if "Doc:" in node:
            G.nodes[node]['color'] = "#38bdf8"
            G.nodes[node]['size'] = 30
        else:
            G.nodes[node]['color'] = "#22c55e"
            G.nodes[node]['size'] = 15

    net.from_nx(G)

    net.set_options("""
    var options = {
      "nodes": {
        "font": {
          "size": 18,
          "face": "Arial"
        }
      },
      "edges": {
        "color": {
          "color": "#1d4ed8"
        },
        "smooth": {
          "type": "dynamic"
        }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 200,
        "navigationButtons": true,
        "zoomView": true,
        "dragView": true
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -20000,
          "centralGravity": 0.02,
          "springLength": 300
        },
        "stabilization": {
          "enabled": true,
          "iterations": 1000
        }
      }
    }
    """)

    output_path = os.path.join(APP_DIR, "graphs", "policy_kg.html")
    net.save_graph(output_path)
    return output_path




Writing knowledge_graph.py


## Cell 9 — `nlp_engine.py` (Unchanged)

In [ ]:
%%writefile nlp_engine.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
import vector_store

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRANSLATOR_ID = "facebook/nllb-200-distilled-600M"

# Globally cached in Streamlit
model = None
tokenizer = None
translator_model = None
translator_tokenizer = None

def init_model():
    global model, tokenizer, translator_model, translator_tokenizer
    if model is None:
        print(f"Loading {MODEL_ID} in 4-bit Quantization via BitsAndBytes...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            torch_dtype=torch.float16
        )

        print(f"Loading {TRANSLATOR_ID} for extreme speed translation...")
        translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ID)
        translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_ID, device_map="auto",torch_dtype=torch.float16)
        print("Models Loaded successfully!")

# BCP-47 Code mappings for NLLB-200
LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

def translate_fast(text, source_lang, target_lang):
    if translator_model is None:
        init_model()
    if source_lang == target_lang: return text

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    translator_tokenizer.src_lang = src_code
    inputs = translator_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(translator_model.device)

    # Fix for AttributeError: Use convert_tokens_to_ids instead of lang_code_to_id
    tgt_token_id = translator_tokenizer.convert_tokens_to_ids(tgt_code)
    outputs = translator_model.generate(
        **inputs,
        forced_bos_token_id=tgt_token_id,
        max_length=512,
        num_beams=2
    )
    return translator_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_english_response(prompt_text):
    if model is None: init_model()
    messages = [
        {"role": "system", "content": "You are the Public Policy Compass AI utilizing RAG. Give accurate, highly concise answers in English based closely on the context."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=200,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def answer_policy_question(native_question, target_lang="English", simplify=False, top_k=5):
    # 1. Bi-directional matching: Translate their Native question into English FIRST so FAISS can search the Vector DB properly
    english_query = translate_fast(native_question, target_lang, "English")

    # 2. Search FAISS Vectors (Multiple distinct sources)
    docs = vector_store.search_documents(english_query, top_k=top_k)
    if not docs: docs_context = "No relevant policies found in Vector DB."
    else: docs_context = "\n\n".join([f"Snippet from {d['filename']}:\n{d['content']}" for d in docs])

    prompt = f"Context:\n{docs_context}\n\nQuestion: {english_query}\nAnswer the question using the context. Be direct."
    if simplify: prompt += " Simplify the policy language so a middle-school student can immediately understand it."

    # 3. LLM Generates English Answer
    eng_ans = generate_english_response(prompt)

    # 4. NLLB perfectly translates English Answer back to Native Language
    final_ans = translate_fast(eng_ans, "English", target_lang)
    return final_ans, docs

def summarize_document(text, target_lang="English"):
    eng_sum = generate_english_response(f"Summarize this policy into 3 highly concise bullet points:\n\n{text[:3000]}")
    return translate_fast(eng_sum, "English", target_lang)




Writing nlp_engine.py


## Cell 10 — `profile_page.py` ✨ NEW — Milestone 4 User Dashboard

Fully adapted to teammates' `db.py`, `auth.py`, and `config.py`.

4 tabs: **Overview · Avatar · Change Email · Change Password**


In [ ]:
%%writefile profile_page.py
# ============================================================
# profile_page.py — PolicyNav User Dashboard (Milestone 4)
# Adapted to teammates' db.py / auth.py / config.py
# ============================================================
import streamlit as st
import base64
import io
import re
from PIL import Image
import db as db_module
from auth import verify_text, hash_text, validate_password, send_otp_email, generate_otp, generate_token
from config import PASSWORD_HISTORY_COUNT
import json


# ── Public helper — used by app.py sidebar ──────────────────
def avatar_html(b64, initials, size=48):
    """Returns circular avatar HTML — photo if set, initials if not."""
    if b64:
        return (
            f"<img src='{b64}' style='width:{size}px;height:{size}px;"
            f"border-radius:50%;object-fit:cover;"
            f"border:2px solid #ef4444;display:block;'/>"
        )
    return (
        f"<div style='width:{size}px;height:{size}px;border-radius:50%;"
        f"background:linear-gradient(135deg,#ef4444,#f87171);"
        f"display:flex;align-items:center;justify-content:center;"
        f"font-size:{max(size//3,10)}px;font-weight:700;color:white;"
        f"flex-shrink:0;'>{initials}</div>"
    )


# ── Private helpers ──────────────────────────────────────────
def _valid_email(email):
    return bool(re.fullmatch(r'^[^@\s]+@[^@\s]+\.[^@\s]+$', email))


def _pw_strength_html(pw):
    """Returns coloured HTML badge for password strength."""
    valid, msg = validate_password(pw)
    if valid:
        return "<span style='color:#34d399;font-weight:600;font-size:13px;'>✓ Strong</span>"
    if len(pw) >= 6:
        return f"<span style='color:#fbbf24;font-weight:600;font-size:13px;'>◑ Medium — {msg}</span>"
    return f"<span style='color:#f87171;font-weight:600;font-size:13px;'>✗ Weak — {msg}</span>"


def _process_avatar(uploaded_file):
    """Crops to square, resizes to 200x200, returns Base64 data URI."""
    try:
        img  = Image.open(uploaded_file).convert('RGB')
        w, h = img.size
        m    = min(w, h)
        img  = img.crop(((w-m)//2, (h-m)//2, (w-m)//2+m, (h-m)//2+m))
        img  = img.resize((200, 200), Image.LANCZOS)
        buf  = io.BytesIO()
        img.save(buf, format='JPEG', quality=85)
        b64  = base64.b64encode(buf.getvalue()).decode('utf-8')
        return f'data:image/jpeg;base64,{b64}', None
    except Exception as e:
        return None, str(e)


def _avatar_html(avatar_b64, initials, size=90):
    """Returns HTML for avatar — photo if set, initials circle otherwise."""
    if avatar_b64:
        return (
            f"<img src='{avatar_b64}' style='width:{size}px;height:{size}px;"
            f"border-radius:50%;object-fit:cover;border:3px solid #ef4444;"
            f"box-shadow:0 0 20px rgba(239,68,68,0.4);'/>"
        )
    return (
        f"<div style='width:{size}px;height:{size}px;border-radius:50%;"
        f"background:linear-gradient(135deg,#ef4444,#f87171);display:flex;"
        f"align-items:center;justify-content:center;font-size:{size//3}px;"
        f"font-weight:700;color:white;"
        f"box-shadow:0 0 20px rgba(239,68,68,0.4);flex-shrink:0;'>{initials}</div>"
    )


# ── Tab 1: Overview ──────────────────────────────────────────
def _tab_overview(email, profile):
    username    = profile.get('username', email.split('@')[0])
    initials    = username[:2].upper()
    avatar_html = _avatar_html(profile.get('avatar'), initials, size=90)

    st.markdown(f"""
    <div style='background:linear-gradient(135deg,#111827,#1a1f35);
                border:1px solid #1f2937;border-radius:16px;
                padding:32px;display:flex;align-items:center;
                gap:24px;margin-bottom:24px;'>
        {avatar_html}
        <div style='flex:1;'>
            <div style='color:#f1f5f9;font-size:22px;font-weight:700;'>{username}</div>
            <div style='color:#475569;font-size:14px;margin-top:4px;'>{email}</div>
        </div>
        <div style='text-align:right;'>
            <div style='color:#34d399;font-size:12px;font-weight:600;'>● Active</div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)
    last = profile['last_active'][:10] if profile['last_active'] != 'No activity yet' else '—'
    for col, icon, label, value, accent in [
        (c1, '⚡', 'Total Activities',  profile['total_activities'], '#ef4444'),
        (c2, '💬', 'Feedbacks Given',   profile['total_feedback'],   '#f87171'),
        (c3, '🕐', 'Last Active',       last,                        '#fca5a5'),
    ]:
        col.markdown(f"""
        <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;
                    padding:20px;text-align:center;border-top:3px solid {accent};'>
            <div style='font-size:24px;margin-bottom:8px;'>{icon}</div>
            <div style='font-size:26px;font-weight:700;color:{accent};'>{value}</div>
            <div style='color:#64748b;font-size:12px;margin-top:4px;'>{label}</div>
        </div>
        """, unsafe_allow_html=True)


# ── Tab 2: Avatar ────────────────────────────────────────────
def _tab_avatar(email):
    st.markdown('<br>', unsafe_allow_html=True)
    current  = db_module.get_avatar(email)
    initials = (db_module.get_user_by_email(email) or [None, email])[1][:2].upper()

    col_prev, col_up = st.columns([1, 2])
    with col_prev:
        st.markdown("<div style='color:#94a3b8;font-size:13px;font-weight:600;margin-bottom:12px;'>Current Avatar</div>", unsafe_allow_html=True)
        st.markdown(
            f"<div style='display:flex;justify-content:center;padding:20px;'>{_avatar_html(current, initials, 120)}</div>",
            unsafe_allow_html=True
        )
        if current:
            if st.button('🗑️ Remove Photo', use_container_width=True, key='remove_avatar'):
                db_module.save_avatar(email, None)
                st.toast('Avatar removed.', icon='🗑️')
                st.rerun()

    with col_up:
        st.markdown("<div style='color:#94a3b8;font-size:13px;font-weight:600;margin-bottom:12px;'>Upload New Photo</div>", unsafe_allow_html=True)
        st.markdown("""
        <div style='background:#0f172a;border:1px solid #1f2937;border-radius:10px;
                    padding:14px;margin-bottom:16px;color:#64748b;font-size:12px;line-height:1.8;'>
            📐 Max size: <strong style='color:#94a3b8;'>5 MB</strong><br>
            🖼️ Formats: <strong style='color:#94a3b8;'>JPG, PNG, WEBP</strong><br>
            ✂️ Auto-cropped to a <strong style='color:#94a3b8;'>200×200 px square</strong>
        </div>
        """, unsafe_allow_html=True)
        uploaded = st.file_uploader('Choose image', type=['jpg','jpeg','png','webp'],
                                    key='avatar_uploader', label_visibility='collapsed')
        if uploaded:
            if uploaded.size > 5 * 1024 * 1024:
                st.error('⚠️ File too large. Max 5 MB.')
            else:
                st.markdown('**Preview:**')
                st.image(uploaded, width=150)
                if st.button('💾 Save as Avatar', use_container_width=True, key='save_avatar'):
                    with st.spinner('Processing...'):
                        b64, err = _process_avatar(uploaded)
                    if err:
                        st.error(f'❌ {err}')
                    else:
                        db_module.save_avatar(email, b64)
                        st.success('✅ Avatar updated!')
                        st.balloons()
                        st.rerun()


# ── Tab 3: Change Email ──────────────────────────────────────
def _tab_change_email(email):
    """2-step: verify current password → OTP sent to new email."""
    st.markdown('<br>', unsafe_allow_html=True)

    if 'email_change_step' not in st.session_state:
        st.session_state.email_change_step = 1

    step = st.session_state.email_change_step

    # Step indicator
    c1 = '#ef4444' if step == 1 else '#1f2937'
    c2 = '#ef4444' if step == 2 else '#1f2937'
    t1 = '#f1f5f9' if step == 1 else '#475569'
    t2 = '#f1f5f9' if step == 2 else '#475569'
    st.markdown(f"""
    <div style='display:flex;align-items:center;gap:12px;margin-bottom:28px;
                padding:16px;background:#0f172a;border-radius:12px;'>
        <div style='background:{c1};border-radius:50%;width:28px;height:28px;
                    display:flex;align-items:center;justify-content:center;
                    font-size:12px;font-weight:700;color:white;'>1</div>
        <span style='color:{t1};font-size:13px;'>Enter New Email</span>
        <div style='flex:1;height:1px;background:#1f2937;'></div>
        <div style='background:{c2};border-radius:50%;width:28px;height:28px;
                    display:flex;align-items:center;justify-content:center;
                    font-size:12px;font-weight:700;color:white;'>2</div>
        <span style='color:{t2};font-size:13px;'>Verify OTP</span>
    </div>
    """, unsafe_allow_html=True)

    if step == 1:
        st.markdown("<div style='color:#64748b;font-size:13px;margin-bottom:16px;'>Enter your new email and current password. An OTP will be sent to the <strong style='color:#fca5a5;'>new email</strong>.</div>", unsafe_allow_html=True)
        new_email  = st.text_input('New email address', placeholder='new@example.com', key='ce_new')
        curr_pass  = st.text_input('Current password',  type='password',               key='ce_pass')
        st.markdown('<br>', unsafe_allow_html=True)

        if st.button('Send Verification Code →', use_container_width=True, key='ce_send'):
            new_email = new_email.strip()
            if not new_email:                                  st.error('Enter a new email.'); return
            if not _valid_email(new_email):                   st.error('Invalid email format.'); return
            if new_email.lower() == email.lower():            st.error('New email must differ from current.'); return
            if db_module.get_user_by_email(new_email):        st.error('That email is already registered.'); return
            if not curr_pass:                                  st.error('Enter your current password.'); return

            user = db_module.get_user_by_email(email)
            if not user or not verify_text(curr_pass, user[3]):
                st.error('❌ Incorrect current password.'); return

            try:
                otp = generate_otp()
                send_otp_email(new_email, otp)
                st.session_state.profile_otp          = otp
                st.session_state.profile_pending_email = new_email
                db_module.store_pending_email(email, new_email)
                st.session_state.email_change_step = 2
                st.success(f'✅ Code sent to {new_email}')
                st.rerun()
            except Exception as ex:
                st.error(f'Could not send email: {ex}')

    elif step == 2:
        pending = st.session_state.get('profile_pending_email', 'your new email')
        st.markdown(f"<div style='color:#64748b;font-size:13px;margin-bottom:16px;'>Enter the 6-digit code sent to <strong style='color:#ef4444;'>{pending}</strong>.</div>", unsafe_allow_html=True)
        otp_in = st.text_input('Verification code', max_chars=6, placeholder='000000', key='ce_otp')
        st.markdown('<br>', unsafe_allow_html=True)

        col_v, col_b = st.columns(2)
        if col_v.button('✅ Confirm Email Change', use_container_width=True, key='ce_verify'):
            if otp_in == st.session_state.get('profile_otp', ''):
                new_confirmed = db_module.confirm_email_update(email)
                if new_confirmed:
                    st.session_state.token = generate_token(new_confirmed)
                    st.session_state.email_change_step = 1
                    st.success('✅ Email updated! Please re-login.')
                    st.balloons()
                    import time; time.sleep(1.5)
                    st.session_state.token = None
                    st.session_state.page  = 'Login'
                    st.rerun()
                else:
                    st.error('Update failed. Please try again.')
            else:
                st.error('❌ Incorrect OTP. Please try again.')

        if col_b.button('← Go Back', use_container_width=True, key='ce_back'):
            st.session_state.email_change_step = 1
            st.rerun()


# ── Tab 4: Change Password ───────────────────────────────────
def _tab_change_password(email):
    st.markdown('<br>', unsafe_allow_html=True)
    st.markdown("""
    <div style='background:#0f172a;border:1px solid #1f2937;border-radius:10px;
                padding:14px;margin-bottom:20px;color:#64748b;font-size:12px;line-height:1.8;'>
        🔒 Requirements:<br>
        &nbsp;&nbsp;• Minimum <strong style='color:#94a3b8;'>8 characters</strong><br>
        &nbsp;&nbsp;• At least 1 <strong style='color:#94a3b8;'>uppercase</strong>, 1 <strong style='color:#94a3b8;'>lowercase</strong>, 1 <strong style='color:#94a3b8;'>number</strong><br>
        &nbsp;&nbsp;• Cannot reuse a <strong style='color:#94a3b8;'>previous password</strong>
    </div>
    """, unsafe_allow_html=True)

    curr = st.text_input('Current password',     type='password', key='cp_curr')
    new  = st.text_input('New password',          type='password', key='cp_new')
    if new:
        st.markdown(_pw_strength_html(new), unsafe_allow_html=True)
    conf = st.text_input('Confirm new password', type='password', key='cp_conf')
    if conf and new:
        match_html = ("<span style='color:#34d399;font-size:13px;'>✓ Passwords match</span>"
                      if conf == new else
                      "<span style='color:#f87171;font-size:13px;'>✗ Passwords do not match</span>")
        st.markdown(match_html, unsafe_allow_html=True)

    st.markdown('<br>', unsafe_allow_html=True)
    if st.button('🔐 Update Password', use_container_width=True, key='cp_submit'):
        if not curr:      st.error('Enter your current password.'); return
        if not new:       st.error('Enter a new password.'); return
        if not conf:      st.error('Confirm your new password.'); return
        if new != conf:   st.error('New passwords do not match.'); return

        user = db_module.get_user_by_email(email)
        if not user or not verify_text(curr, user[3]):
            st.error('❌ Incorrect current password.'); return

        valid, msg = validate_password(new)
        if not valid:
            st.error(f'Password too weak — {msg}'); return

        history = json.loads(user[8] or '[]')
        for old_hash in history:
            if verify_text(new, old_hash):
                st.error('⚠️ Cannot reuse a previous password.'); return

        new_hash = hash_text(new)
        db_module.update_password(email, new_hash)
        history.insert(0, new_hash)
        history = history[:PASSWORD_HISTORY_COUNT]
        db_module.update_password_history(email, json.dumps(history))
        st.success('✅ Password updated successfully!')
        st.balloons()


# ── Tab 5: Settings ─────────────────────────────────────────
def _tab_settings(email):
    """Full app settings — language, notifications, display, RAG preferences."""
    import db as _db

    LANGUAGES = [
        'English', 'Hindi', 'Tamil', 'Telugu', 'Kannada',
        'Malayalam', 'Marathi', 'Bengali', 'Gujarati',
        'Punjabi', 'Urdu', 'Odia',
    ]

    # Load current saved settings
    s           = _db.get_user_settings(email)
    cur_lang    = s.get('app_language',     'English')
    cur_notif   = s.get('notifications',    True)
    cur_ts      = s.get('show_timestamps',  True)
    cur_rpp     = s.get('results_per_page', 5)

    st.markdown('<br>', unsafe_allow_html=True)

    # ── Section 1: Language ──────────────────────────────────
    st.markdown("""
    <div style='background:#0f172a;border:1px solid #1f2937;
                border-radius:14px;padding:20px;margin-bottom:16px;'>
        <div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'>
            <div style='width:34px;height:34px;background:#ef4444;border-radius:8px;
                        display:flex;align-items:center;justify-content:center;
                        font-size:16px;'>🌐</div>
            <div>
                <div style='color:#f1f5f9;font-weight:700;font-size:15px;'>
                    Language & Region</div>
                <div style='color:#475569;font-size:12px;'>
                    Set your preferred language for AI responses</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    lang_idx      = LANGUAGES.index(cur_lang) if cur_lang in LANGUAGES else 0
    selected_lang = st.selectbox(
        'Application Language',
        LANGUAGES,
        index=lang_idx,
        key='s_lang',
        help='AI responses in RAG Search and Summarizer will use this language'
    )

    # Show language badge grid
    st.markdown(
        "<div style='color:#64748b;font-size:12px;"
        "margin:8px 0 6px;'>Supported languages:</div>",
        unsafe_allow_html=True)
    cols = st.columns(6)
    for idx, lang in enumerate(LANGUAGES):
        active = lang == selected_lang
        bg     = '#ef4444' if active else '#111827'
        col_t  = '#ffffff' if active else '#64748b'
        border = '#ef4444' if active else '#1f2937'
        cols[idx % 6].markdown(
            f"<div style='background:{bg};border:1px solid {border};"
            f"border-radius:6px;padding:4px 6px;margin-bottom:4px;"
            f"color:{col_t};font-size:11px;text-align:center;'>{lang}</div>",
            unsafe_allow_html=True)

    st.markdown('<br>', unsafe_allow_html=True)

    # ── Section 2: Notifications ─────────────────────────────
    st.markdown("""
    <div style='background:#0f172a;border:1px solid #1f2937;
                border-radius:14px;padding:20px;margin-bottom:16px;'>
        <div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'>
            <div style='width:34px;height:34px;background:#f59e0b;border-radius:8px;
                        display:flex;align-items:center;justify-content:center;
                        font-size:16px;'>🔔</div>
            <div>
                <div style='color:#f1f5f9;font-weight:700;font-size:15px;'>
                    Notifications</div>
                <div style='color:#475569;font-size:12px;'>
                    Control in-app alerts and toast messages</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    notif_on = st.toggle(
        'Enable notifications',
        value=cur_notif,
        key='s_notif',
        help='Show success/error toast messages after actions'
    )
    st.caption(
        '✅ Enabled — you will see toast alerts for login, feedback, '
        'password changes etc.' if notif_on
        else '🔕 Disabled — toast messages will be suppressed.'
    )

    st.markdown('<br>', unsafe_allow_html=True)

    # ── Section 3: Display Preferences ──────────────────────
    st.markdown("""
    <div style='background:#0f172a;border:1px solid #1f2937;
                border-radius:14px;padding:20px;margin-bottom:16px;'>
        <div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'>
            <div style='width:34px;height:34px;background:#6366f1;border-radius:8px;
                        display:flex;align-items:center;justify-content:center;
                        font-size:16px;'>🖥️</div>
            <div>
                <div style='color:#f1f5f9;font-weight:700;font-size:15px;'>
                    Display Preferences</div>
                <div style='color:#475569;font-size:12px;'>
                    Customise how information is shown to you</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    show_ts = st.toggle(
        'Show timestamps in activity history',
        value=cur_ts,
        key='s_ts',
        help='Display date and time for each activity in your history page'
    )

    st.markdown('<br>', unsafe_allow_html=True)

    # ── Section 4: RAG Search Preferences ───────────────────
    st.markdown("""
    <div style='background:#0f172a;border:1px solid #1f2937;
                border-radius:14px;padding:20px;margin-bottom:16px;'>
        <div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'>
            <div style='width:34px;height:34px;background:#22c55e;border-radius:8px;
                        display:flex;align-items:center;justify-content:center;
                        font-size:16px;'>🔍</div>
            <div>
                <div style='color:#f1f5f9;font-weight:700;font-size:15px;'>
                    Search Preferences</div>
                <div style='color:#475569;font-size:12px;'>
                    Configure RAG search behaviour</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    rpp = st.select_slider(
        'Number of document results per search',
        options=[3, 5, 7, 10],
        value=cur_rpp if cur_rpp in [3, 5, 7, 10] else 5,
        key='s_rpp',
        help='How many policy document chunks the AI uses to answer your question'
    )
    st.caption(
        f'Currently set to **{rpp}** results. '
        'More results = more context but slightly slower response.'
    )

    st.markdown('<br>', unsafe_allow_html=True)

    # ── Save button ──────────────────────────────────────────
    col_save, col_reset = st.columns([3, 1])
    with col_save:
        if st.button(
            '💾 Save All Settings',
            use_container_width=True,
            key='save_all_settings',
            type='primary'
        ):
            _db.save_user_settings(
                email,
                app_language     = selected_lang,
                notifications    = notif_on,
                show_timestamps  = show_ts,
                results_per_page = rpp,
            )
            # Store in session so other pages can read them immediately
            st.session_state['app_language']     = selected_lang
            st.session_state['show_timestamps']  = show_ts
            st.session_state['notifications']    = notif_on
            st.session_state['results_per_page'] = rpp
            st.success('✅ Settings saved successfully!')
            st.balloons()
            st.rerun()

    with col_reset:
        if st.button(
            '↺ Reset',
            use_container_width=True,
            key='reset_settings',
            help='Reset all settings to default values'
        ):
            _db.save_user_settings(
                email,
                app_language='English',
                notifications=True,
                show_timestamps=True,
                results_per_page=5,
            )
            st.session_state['app_language']     = 'English'
            st.session_state['show_timestamps']  = True
            st.session_state['notifications']    = True
            st.session_state['results_per_page'] = 5
            st.info('Settings reset to defaults.')
            st.rerun()


# ── Public entry point ───────────────────────────────────────
def render_profile_page(email):
    """
    Main entry point — called from app.py:
        import profile_page
        profile_page.render_profile_page(email)
    """
    st.markdown("""
    <div style='margin-bottom:1.5rem;'>
        <div style='display:flex;align-items:center;gap:10px;'>
            <span style='font-size:24px;'>👤</span>
            <div>
                <h2 style='margin:0;color:#fca5a5;'>My Profile</h2>
                <p style='margin:2px 0 0;color:#475569;font-size:13px;'>
                    Manage your account, avatar and security settings
                </p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)
    st.divider()

    profile = db_module.get_user_profile(email)
    if not profile:
        st.error('Could not load profile. Please log out and back in.')
        return

    tab1, tab2, tab3, tab4, tab5 = st.tabs([
        '👤  Overview',
        '🖼️  Avatar',
        '📧  Change Email',
        '🔒  Change Password',
        '⚙️  Settings',
    ])
    with tab1: _tab_overview(email, profile)
    with tab2: _tab_avatar(email)
    with tab3: _tab_change_email(email)
    with tab4: _tab_change_password(email)
    with tab5: _tab_settings(email)


Writing profile_page.py


## Cell 11 — `app.py` (Teammates' code + Profile page integrated)

Changes from teammates' original:
1. `import profile_page` added at top
2. `'👤 Profile'` added to user sidebar
3. `elif page == 'Profile': profile_tab()` added to router
4. `profile_tab()` function added


In [ ]:
%%writefile app.py
import plotly.graph_objects as go
import os
import time
import PyPDF2
from readability import ReadabilityAnalyzer
import streamlit as st
import json
from datetime import datetime, timedelta
from db import *
from auth import *
from config import *
import base64
import vector_store
import nlp_engine
import knowledge_graph
import pandas as pd
import plotly.express as px
from streamlit_option_menu import option_menu
import profile_page  # Milestone 4 — User Dashboard

st.set_page_config(page_title="PolicyNav Milestone_3", layout="centered")
st.markdown("""
<style>
/* Professional LLM Aesthetics */
.stApp {
    background-color: #0b0f19;
    color: #e2e8f0;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
}
div[data-testid="stSidebar"] {
    background-color: #111827 !important;
    border-right: 1px solid #1f2937;
}
div[data-testid="stChatMessage"] {
    background-color: #1e293b;
    border-radius: 12px;
    border: 1px solid #334155;
    padding: 16px;
    margin-bottom: 16px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
}
.stButton>button {
    background-color: #ef4444;
    color: white;
    border-radius: 8px;
    font-weight: 500;
    border: 1px solid #dc2626;
    height: 42px;
    transition: all 0.2s ease;
    box-shadow: 0 1px 2px 0 rgba(0, 0, 0, 0.05);
}
.stButton>button:hover {
    background-color: #dc2626;
    border-color: #b91c1c;
    color: white;
}
div[data-baseweb="input"] > div {
    background-color: #1e293b;
    border: 1px solid #334155;
    border-radius: 8px;
    color: #f8fafc;
}
div[data-baseweb="input"] > div:hover {
    border-color: #ef4444;
}
h1, h2, h3, h4, h5, h6 {
    color: #f8fafc;
    font-weight: 600;
}
/* Title generic styling */
.project-title {
    font-weight: 800;
    font-size: 28px;
    color: #ffffff;
    padding-bottom: 16px;
    border-bottom: 1px solid #1f2937;
    margin-bottom: 24px;
}
.project-title span { color: #ef4444; }
</style>
""", unsafe_allow_html=True)


# --- NEW GENERIC HEADER ---
def render_header():
    st.markdown("<div class='project-title'><span>Public Policy</span> Navigation</div>", unsafe_allow_html=True)

init_db()
# ---------- LOAD AI MODELS ----------
@st.cache_resource
def load_and_cache_models():
    nlp_engine.init_model()
    vector_store.get_embedder()

    return True

with st.spinner("Initializing AI Core..."):
    load_and_cache_models()

# --- REUSABLE UI COMPONENTS ---
def render_feedback_ui(section_name, generated_text, unique_key):

    payload = verify_token(st.session_state.token)

    if not payload:
        return

    user_email = payload["email"]

    with st.expander(f"📝 Provide Feedback for {section_name}"):

        col1, col2 = st.columns([1,4])

        with col1:
            rating = st.radio(
                "Rating (1-5)",
                [1,2,3,4,5],
                horizontal=True,
                key=f"r_{unique_key}"
            )

        with col2:
            comments = st.text_input(
                "Comments (optional)",
                key=f"c_{unique_key}"
            )

        if st.button("Submit Feedback", key=f"fb_{unique_key}"):

            submit_feedback(
                user_email,
                section_name,
                rating,
                comments
            )

            st.toast("✅ Thank you for your feedback!")
# ------------------------------

# set_bg() removed — dark theme handled by CSS above

# ---------- SESSION STATE ----------
if "token" not in st.session_state:
    st.session_state.token = None

if "page" not in st.session_state:
    st.session_state.page = "Login"

if "reset_stage" not in st.session_state:
    st.session_state.reset_stage = 0

# ── Load saved user settings into session on every run ──────
def _load_user_settings():
    """Reads saved settings from DB once per login session."""
    if (st.session_state.token and
            st.session_state.token != "ADMIN_OVERRIDE" and
            not st.session_state.get("settings_loaded", False)):
        try:
            payload = verify_token(st.session_state.token)
            if payload:
                s = get_user_settings(payload["email"])
                st.session_state["app_language"]     = s.get("app_language",     "English")
                st.session_state["notifications"]    = s.get("notifications",    True)
                st.session_state["show_timestamps"]  = s.get("show_timestamps",  True)
                st.session_state["results_per_page"] = s.get("results_per_page", 5)
                st.session_state["settings_loaded"]  = True
        except:
            pass

_load_user_settings()

# ---------- NAVIGATION ----------
def go_to(page):
    st.session_state.page = page
    st.rerun()

# ---------- LOGIN ----------
def login_page():
    st.title("PolicyNav – Login")
    render_header()

    with st.form("user_login_form"):
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        left, center, right = st.columns([1.5, 1.5, 5])

        login_btn = left.form_submit_button("Login")
        signup_btn = center.form_submit_button("Signup")
        forgot_btn = right.form_submit_button("Forgot Password")

        if login_btn:
            if not email:
                st.toast("Email required", icon="⚠️")
                return
            if not password:
                st.toast("Password required", icon="⚠️")
                return

            _ae = os.getenv("ADMIN_EMAIL_ID"); _ap = os.getenv("ADMIN_PASSWORD")
            if email == _ae and password == _ap:
                st.session_state.token = "ADMIN_OVERRIDE"
                st.session_state.is_admin = True
                go_to("AdminDashboard")
                return

            user = get_user_by_email(email)
            if not user:
                st.toast("Email not registered", icon="❌")
                return

            failed_attempts = user[6]
            lock_until = user[7]

            is_admin = len(user) > 9 and user[9] == 1
            st.session_state.is_admin = is_admin

            if lock_until and datetime.utcnow() < datetime.fromisoformat(lock_until):
                st.toast("Account locked. Try again later.", icon="🔒")
                return

            if not verify_text(password, user[3]):
                failed_attempts += 1
                if failed_attempts >= MAX_LOGIN_ATTEMPTS:
                    lock_time = datetime.utcnow() + timedelta(minutes=LOCK_TIME_MINUTES)
                    update_login_attempts(email, failed_attempts, lock_time.isoformat())
                    st.toast("Account locked for 5 minutes", icon="🔒")
                else:
                    update_login_attempts(email, failed_attempts)
                    st.toast("Invalid credentials", icon="❌")
                return

            update_login_attempts(email, 0, None)

            st.session_state.token = generate_token(email)
            if is_admin:
                go_to("AdminDashboard")
            else:
                go_to("Dashboard")

        if signup_btn:
            go_to("Signup")

        if forgot_btn:
            go_to("Forgot")

# ---------- OTP ----------
def otp_page():
    st.title("OTP Verification")
    render_header()

    entered = st.text_input("Enter OTP")

    if st.button("Verify OTP"):

        if "otp_time" not in st.session_state:
            st.toast("OTP expired. Please login again.", icon="⚠️")
            go_to("Login")
            return

        # Check 10-minute expiry
        if datetime.utcnow() - st.session_state.otp_time > timedelta(minutes=10):
            st.toast("OTP expired. Please login again.", icon="⏰")
            go_to("Login")
            return

        if entered == st.session_state.otp:
            # Forgot password via OTP
            if st.session_state.get("flow") == "forgot_otp":
                go_to("SetNewPassword")
            # Normal login OTP
            else:
                st.session_state.token = generate_token(st.session_state.pending_email)
                go_to("Dashboard")
        else:
            st.toast("Invalid OTP", icon="❌")

    if st.button("⬅ Back to Login"):
        go_to("Login")

# ---------- SIGNUP ----------
def signup_page():
    st.title("Signup")
    render_header()

    username = st.text_input("Username")
    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm = st.text_input("Confirm Password", type="password")

    question = st.selectbox("Security Question", [
        "What is your favorite book?",
        "What is your dream job?"
    ])

    answer = st.text_input("Security Answer")

    if st.button("Create Account"):

        if not validate_email(email):
            st.toast("Invalid email format", icon="⚠️")
            return

        valid, msg = validate_password(password)
        if not valid:
            st.toast(msg, icon="⚠️")
            return

        if password != confirm:
            st.toast("Passwords do not match", icon="⚠️")
            return

        success, msg = create_user(
            username,
            email,
            hash_text(password),
            question,
            hash_text(answer)
        )

        if success:
            st.toast("Account created successfully", icon="✅")
            go_to("Login")
        else:
            st.toast(msg, icon="❌")

    if st.button("⬅ Back to Login"):
        go_to("Login")

# ---------- sidebar function ---------
def render_sidebar():
    # Pages where the menu must not auto-navigate
    NON_MENU = {"Dashboard", "Profile"}

    with st.sidebar:
        st.markdown(
            "<div style='font-weight:800;font-size:22px;color:#ffffff;"
            "margin:4px 0 12px 0;'>"
            "<span style='color:#ef4444;'>Public Policy</span><br>"
            "Navigation</div>",
            unsafe_allow_html=True
        )

        # ── Profile card at the TOP of sidebar ──────────────────
        if st.session_state.token and st.session_state.token != "ADMIN_OVERRIDE":
            _payload = verify_token(st.session_state.token)
            if _payload:
                _prof = get_user_profile(_payload["email"])
                if _prof:
                    _av  = _prof.get("avatar")
                    _un  = _prof.get("username", _payload["email"].split("@")[0])
                    _ini = _un[:2].upper()
                    _ah  = profile_page.avatar_html(_av, _ini, 44)
                    st.markdown(
                        f"<div style='display:flex;align-items:center;gap:10px;"
                        f"background:rgba(255,255,255,0.04);"
                        f"border:1px solid #1f2937;border-radius:12px;"
                        f"padding:10px 12px;margin-bottom:10px;'>"
                        f"{_ah}"
                        f"<div style='overflow:hidden;flex:1;min-width:0;'>"
                        f"<div style='color:#f1f5f9;font-weight:600;font-size:13px;"
                        f"white-space:nowrap;overflow:hidden;text-overflow:ellipsis;'>{_un}</div>"
                        f"<div style='color:#475569;font-size:11px;"
                        f"white-space:nowrap;overflow:hidden;text-overflow:ellipsis;'>{_payload['email']}</div>"
                        f"</div></div>",
                        unsafe_allow_html=True
                    )

        # ── Navigation menu ──────────────────────────────────────
        page_map = {
            "Readability": 0, "RAG": 1,
            "Summarization": 2, "Graph": 3,
            "History": 4, "Profile": 5,
        }
        cur_pg        = st.session_state.page
        current_index = page_map.get(cur_pg, 0) if cur_pg in page_map else 0

        selected = option_menu(
            menu_title=None,
            options=[
                "Readability", "RAG Search", "Summarization",
                "Knowledge Graph", "History", "My Profile", "Logout"
            ],
            icons=[
                "book", "search", "file-text",
                "diagram-3", "clock-history", "person-circle", "box-arrow-right"
            ],
            default_index=current_index,
            styles={
                "container":         {"padding": "0!important", "background-color": "transparent"},
                "icon":              {"color": "#94a3b8", "font-size": "18px"},
                "nav-link":          {"font-size": "15px", "text-align": "left",
                                      "margin": "3px 0", "--hover-color": "#1e293b",
                                      "color": "#cbd5e1", "border-radius": "8px"},
                "nav-link-selected": {"background-color": "#ef4444",
                                      "color": "#ffffff", "font-weight": "600"},
            }
        )

        # Clear state when leaving pages
        if cur_pg == "RAG" and selected != "RAG Search":
            st.session_state.rag_chat = []
        if cur_pg == "Summarization" and selected != "Summarization":
            st.session_state.doc_answer = ""
            if "summary" in st.session_state:
                st.session_state.summary = ""

        # Navigation — clean destination map
        dest = {
            "Readability":     "Readability",
            "RAG Search":      "RAG",
            "Summarization":   "Summarization",
            "Knowledge Graph": "Graph",
            "History":         "History",
            "My Profile":      "Profile",
        }
        if selected in dest and cur_pg != dest[selected]:
            # Guard: don't auto-redirect from Dashboard to Readability on first load
            if not (cur_pg == "Dashboard" and selected == "Readability"):
                go_to(dest[selected])

        if selected == "Logout":
            st.session_state.token = None
            go_to("Login")


# ---------- DASHBOARD ----------
def dashboard_page():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)
    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")
        return

    email = payload["email"]
    user  = get_user_by_email(email)
    prof  = get_user_profile(email)

    # ── Welcome banner ──────────────────────────────────────
    username = user[1] if user else email.split("@")[0]
    initials = username[:2].upper()
    av       = prof.get("avatar") if prof else None
    av_html  = profile_page.avatar_html(av, initials, 64)

    st.markdown(
        f"<div style='background:linear-gradient(135deg,#111827,#1a1f35);"
        f"border:1px solid #1f2937;border-radius:16px;"
        f"padding:24px 28px;display:flex;align-items:center;"
        f"gap:20px;margin-bottom:24px;'>"
        f"{av_html}"
        f"<div>"
        f"<div style='color:#f1f5f9;font-size:22px;font-weight:700;'>"
        f"Welcome back, {username} 👋</div>"
        f"<div style='color:#475569;font-size:13px;margin-top:4px;'>"
        f"Select a feature from the sidebar to get started.</div>"
        f"</div></div>",
        unsafe_allow_html=True
    )

    # ── Quick stats ─────────────────────────────────────────
    if prof:
        c1, c2, c3 = st.columns(3)
        last = prof["last_active"][:10] if prof["last_active"] != "No activity yet" else "—"
        for col, icon, label, val, clr in [
            (c1, "⚡", "Total Activities", prof["total_activities"], "#ef4444"),
            (c2, "💬", "Feedbacks Given",  prof["total_feedback"],   "#f87171"),
            (c3, "🕐", "Last Active",      last,                      "#fca5a5"),
        ]:
            col.markdown(
                f"<div style='background:#111827;border:1px solid #1f2937;"
                f"border-radius:12px;padding:16px;text-align:center;"
                f"border-top:3px solid {clr};'>"
                f"<div style='font-size:20px;margin-bottom:6px;'>{icon}</div>"
                f"<div style='font-size:22px;font-weight:700;color:{clr};'>{val}</div>"
                f"<div style='color:#64748b;font-size:12px;margin-top:4px;'>{label}</div>"
                f"</div>",
                unsafe_allow_html=True
            )

    # ── My Profile button at the bottom ─────────────────────
    st.markdown("<br><br>", unsafe_allow_html=True)
    st.divider()
    col_left, col_mid, col_right = st.columns([1, 2, 1])
    with col_mid:
        if st.button("👤 My Profile", use_container_width=True, key="dash_profile_btn"):
            go_to("Profile")

#--------- RAG search -----------------

def rag_search_tab():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    st.markdown("<h2 style='color: #fca5a5;'>🔍 Advanced RAG Search</h2>", unsafe_allow_html=True)
    st.write("Query the Google Drive Vector Database directly. AI synthesizes answers only from policy context.")

    c1, c2 = st.columns([3, 1])
    # Use saved language preference as default
    _saved_lang = st.session_state.get('app_language', 'English')
    _all_langs  = ['English', 'Hindi', 'Tamil', 'Kannada', 'Telugu', 'Marathi', 'Bengali', 'Malayalam', 'Gujarati', 'Punjabi', 'Urdu', 'Odia']
    _lang_idx   = _all_langs.index(_saved_lang) if _saved_lang in _all_langs else 0
    with c1:
        target_lang = st.selectbox(
            'Response Language (NLLB Pipeline):',
            _all_langs,
            index=_lang_idx,
            key='rag_lang',
            help='Default set from your Settings. Change here for this session only.'
        )
    with c2: simplify = st.toggle("🧠 Simplify Jargon")
    st.divider()

    if "rag_chat" not in st.session_state: st.session_state.rag_chat = []

    for i, msg in enumerate(st.session_state.rag_chat):
        with st.chat_message(msg["role"]): st.markdown(msg["content"])

    if prompt := st.chat_input("Ask a policy question..."):
        st.session_state.rag_chat.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)

        with st.chat_message("assistant"):
            with st.spinner(f"Bi-Directional Vector Lookup & NLLB Extracting > Translating..."):
                t1 = time.time()
                _top_k = st.session_state.get('results_per_page', 5)
                ans, docs = nlp_engine.answer_policy_question(
                    prompt,
                    target_lang=target_lang,
                    simplify=simplify,
                    top_k=_top_k
                )
                t2 = time.time()

                final_txt = f"**{ans}**\n\n---\n*Inference: {round(t2-t1,2)}s | 📚 Sources: {', '.join(list(set([d['filename'] for d in docs]))) if docs else 'None'}*"
                st.markdown(final_txt)

                log_activity(user_email, "RAG Search", prompt, final_txt, language=target_lang)
                st.session_state.rag_chat.append({"role": "assistant", "content": final_txt})
                st.rerun()

    st.divider()
    render_feedback_ui("RAG Search", "General Page Feedback", "rag_global")

# --------- summarizer ----------
def summarization_tab():

    if "summary" not in st.session_state:
        st.session_state.summary = ""

    if "doc_text" not in st.session_state:
        st.session_state.doc_text = ""

    if "doc_answer" not in st.session_state:
        st.session_state.doc_answer = ""

    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    st.markdown("<h2 style='color: #fca5a5;'>📝 Document Summarizer</h2>", unsafe_allow_html=True)
    st.write("Upload a PDF explicitly, or paste raw text to translate and summarize.")

    col1, col2 = st.columns([1,1])

    # -------- LEFT COLUMN (INPUT) --------
    with col1:

        uploaded_file = st.file_uploader("Upload Policy PDF or TXT", type=["pdf","txt"])

        txt = ""

        if uploaded_file is not None:

            if uploaded_file.name.endswith(".pdf"):

                try:
                    reader = PyPDF2.PdfReader(uploaded_file)
                    for page in reader.pages:
                        txt += page.extract_text() + "\n"
                except:
                    pass

            else:
                txt = uploaded_file.read().decode("utf-8")

            st.success("File Processed successfully!")

        txt_area = st.text_area("Or Paste Raw Policy Text:", value=txt, height=200)

    # -------- RIGHT COLUMN (SUMMARY BUTTON) --------
    with col2:

        _saved_lang2 = st.session_state.get('app_language', 'English')
        _all_langs2  = ['English', 'Hindi', 'Tamil', 'Kannada', 'Telugu', 'Marathi', 'Bengali', 'Malayalam', 'Gujarati', 'Punjabi', 'Urdu', 'Odia']
        _lang_idx2   = _all_langs2.index(_saved_lang2) if _saved_lang2 in _all_langs2 else 0
        lang = st.selectbox(
            'Summary Output Language:',
            _all_langs2,
            index=_lang_idx2,
            key='sum_lang',
            help='Default set from your Settings. Change here for this session only.'
        )

        if st.button("Generate Summary", type="primary") and txt_area:

            with st.spinner("Generating Summary..."):

                txt_area = txt_area[:3000]

                summary = nlp_engine.summarize_document(
                    txt_area,
                    target_lang=lang
                )

                st.session_state.summary = summary
                st.session_state.doc_text = txt_area
                st.session_state.doc_answer = ""

                log_activity(user_email,"Summarization","Document Uploaded",summary, language=lang)

    # -------- SHOW SUMMARY --------
    if st.session_state.summary:

        st.info(st.session_state.summary)

        # -------- Q&A SECTION --------
        st.divider()
        st.subheader("❓ Ask Questions About This Document")

        question = st.text_input("Ask a question from the uploaded document")

        if st.button("Get Answer") and question:

            with st.spinner("Analyzing document..."):

                context = st.session_state.doc_text[:2000]

                prompt = f"""
                Context:
                {context}

                Question: {question}

                Answer based only on the context.
                """

                answer = nlp_engine.generate_english_response(prompt)

                st.session_state.doc_answer = answer

        if st.session_state.doc_answer:
            st.success(st.session_state.doc_answer)

    st.divider()
    render_feedback_ui("Summarization","General Page Feedback","sum_global")

#---------- knowledge graph --------
def graph_tab():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    user_email = payload["email"]

    col1, col2, col3 = st.columns(3)

    col1.metric("Documents Indexed", len(vector_store.get_all_documents()))
    col2.metric("Graph Type", "Policy Entity Network")
    col3.metric("NER Engine", "spaCy")

    st.markdown("""
    **Graph Legend**

    🟦 Document
    🟢 Organization
    🩷 Person
    🟠 Law / Policy
    🟡 Location
    """)

    st.markdown("""
    <h2 style='color:#fca5a5; font-weight:700; letter-spacing:1px;'>
    Knowledge Graph Explorer
    </h2>
    """, unsafe_allow_html=True)

    docs = vector_store.get_all_documents()
    if not docs: st.warning("No documents exist in Google Drive."); return

    if st.button("Render Interactive Topology", type="primary", use_container_width=True):
        with st.spinner("Computing Graph Layout..."):
            path = knowledge_graph.generate_interactive_graph(docs)
            if path:
                with open(path, 'r', encoding='utf-8') as f:
                    import streamlit.components.v1 as components
                    components.html(f.read(), height=650)
                log_activity(user_email, "Knowledge Graph", "Generated Graph", "Success", language="English")

    st.divider()
    render_feedback_ui("Knowledge Graph", "General Page Feedback", "kg_global")

#----------- History Tab-----------------

def overall_history_tab():
    render_header()
    st.markdown("<h2 style='color: #fca5a5;'>📜 Global Activity History</h2>", unsafe_allow_html=True)
    st.write("All your RAG searches, summarizations, and metrics from Google Drive.")

    render_sidebar()
    payload = verify_token(st.session_state.token)
    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")
    activities = get_user_activity(payload["email"])
    if not activities:
        st.info("No activity found for your account yet.")
        return

    # Timeline styling
    st.markdown("""
    <style>
    .timeline {
        border-left: 2px solid #334155;
        padding-left: 20px;
        margin-top: 20px;
    }

    .timeline-item {
        margin-bottom: 25px;
        position: relative;
    }

    .timeline-dot {
        width: 12px;
        height: 12px;
        background: #ef4444;
        border-radius: 50%;
        box-shadow: 0 0 10px rgba(239,68,68,0.6);
        position: absolute;
        left: -27px;
        top: 6px;
    }

    .timeline-card {
        background: rgba(30,41,59,0.55);
        border: 1px solid rgba(255,255,255,0.08);
        border-radius: 10px;
        padding: 16px;
    }

    .timeline-card:hover {
        transform: translateX(6px);
        transition: 0.25s;
        box-shadow: 0 12px 30px rgba(0,0,0,0.45);
    }

    .timeline-app {
        font-weight: 600;
        color: #fca5a5;
        font-size: 16px;
    }

    .timeline-time {
        font-size: 12px;
        color: #64748b;
        margin-top: 6px;
    }
    </style>
    """, unsafe_allow_html=True)


    st.markdown('<div class="timeline">', unsafe_allow_html=True)

    for app, user_inp, ai_out, ts in activities:

        st.markdown("""
        <div class="timeline-item">
            <div class="timeline-dot"></div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
            <div class="timeline-card">
                <div class="timeline-app">{app}</div>
                <div style="margin-top:8px;color:#94a3b8;">
                    <b>User:</b> {user_inp}
                </div>
        """, unsafe_allow_html=True)

        st.write(ai_out)  # ← THIS prevents HTML escaping

        st.markdown(f"""
                <div class="timeline-time">
                    ⏱ {ts}
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

# ---------- READABILITY PAGE ----------
def readability_page():
    st.title("📘 Text Readability Analyzer")
    render_header()
    render_sidebar()
    payload = verify_token(st.session_state.token)

    if not payload:
        st.toast("Session expired", icon="❌")
        go_to("Login")

    option = st.radio("Choose Input Type", ["Enter Text", "Upload File"])

    text = ""

    if option == "Enter Text":
        text = st.text_area("Enter text to analyze")

    else:
        uploaded_file = st.file_uploader("Upload TXT or PDF", type=["txt", "pdf"])

        if uploaded_file:

            if uploaded_file.type == "text/plain":
                text = uploaded_file.read().decode("utf-8")

            elif uploaded_file.type == "application/pdf":
                pdf_reader = PyPDF2.PdfReader(uploaded_file)
                for page in pdf_reader.pages:
                    text += page.extract_text() or ""

    if st.button("Analyze Readability"):

        if not text.strip():
            st.toast("No text found to analyze", icon="⚠️")
            return

        analyzer = ReadabilityAnalyzer(text)
        metrics = analyzer.get_all_metrics()

        st.subheader("Results")
        st.success(
        f"Overall Grade Level: {metrics['Overall Grade Level']:.2f}"
        )

        avg_grade = metrics["Overall Grade Level"]
        if avg_grade <= 6:
            level, color = 'Beginner', '#28a745'
        elif avg_grade <= 10:
            level, color = 'Intermediate', '#17a2b8'
        elif avg_grade <= 14:
            level, color = 'Advanced', '#ffc107'
        else:
            level, color = 'Expert', '#dc3545'
        st.markdown(f"<div style='font-weight:600;color:{color};'>Level: {level}</div>", unsafe_allow_html=True)

        st.subheader("Text Statistics")

        c1, c2, c3, c4, c5 = st.columns(5)

        c1.metric("Sentences", metrics["Total Sentences"])
        c2.metric("Words", metrics["Total Words"])
        c3.metric("Syllables", metrics["Total Syllables"])
        c4.metric("Complex Words", metrics["Complex Words"])
        c5.metric("Characters", metrics["Total Characters"])

        col1, col2 = st.columns(2)

        with col1:
          st.plotly_chart(create_gauge(metrics["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100))
          st.caption(f"Flesch Reading Ease Value: {metrics['Flesch Reading Ease']:.2f}")

          st.plotly_chart(create_gauge(metrics["SMOG Index"], "SMOG Index", 0, 20))
          st.caption(f"SMOG Index Value: {metrics['SMOG Index']:.2f}")

          st.plotly_chart(create_gauge(metrics["Coleman-Liau"], "Coleman-Liau Index", 0, 20))
          st.caption(f"Coleman-Liau Value: {metrics['Coleman-Liau']:.2f}")

        with col2:
          st.plotly_chart(create_gauge(metrics["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20))
          st.caption(f"Flesch-Kincaid Grade Value: {metrics['Flesch-Kincaid Grade']:.2f}")

          st.plotly_chart(create_gauge(metrics["Gunning Fog"], "Gunning Fog", 0, 20))
          st.caption(f"Gunning Fog Value: {metrics['Gunning Fog']:.2f}")

    st.divider()
    render_feedback_ui("Readability", "General Page Feedback", "read_global")

    if st.button("⬅ Back to Dashboard"):
        go_to("Dashboard")

# ------------ Gauge --------
def create_gauge(value, title, min_val, max_val):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title},
        gauge={'axis': {'range': [min_val, max_val]}}
    ))
    fig.update_layout(height=250)
    return fig

# ---------- ADMIN ----------
def admin_dashboard():
    with st.sidebar:
        st.markdown("<div style='font-weight:800;font-size:24px;color:#ffffff;margin:2px 0 20px 0;'><span style='color:#ef4444;'>PolicyNav</span><br>Admin Dashboard</div>", unsafe_allow_html=True)

        selected = option_menu(
            menu_title=None,
            options=["User Control", "Activity Tracking", "Data Visualization", "Feedback Analysis", "Data Export", "Logout"],
            icons=["people", "activity", "bar-chart", "chat-square-quote", "download", "box-arrow-right"],
            default_index=0,
            styles={
                "container": {"padding": "0!important", "background-color": "transparent"},
                "icon": {"color": "#94a3b8", "font-size": "18px"},
                "nav-link": {"font-size": "16px", "text-align": "left", "margin": "4px 0", "--hover-color": "#1e293b", "color": "#cbd5e1", "border-radius": "8px"},
                "nav-link-selected": {"background-color": "#ef4444", "color": "#ffffff", "font-weight": "600"},
            }
        )

        if selected == "Logout":
            st.session_state.token              = None
            st.session_state.is_admin           = False
            st.session_state['settings_loaded'] = False
            go_to("Login")

    st.title("🛠 Admin Dashboard")
    render_header()

    # Check Admin Authorization
    if st.session_state.get("token") != "ADMIN_OVERRIDE" and not st.session_state.get("is_admin", False):
         st.error("Unauthorized access.")
         st.stop()

    if selected == "User Control":
        st.subheader("👥 User Management")
        users = get_all_users()
        if not users:
            st.info("No users found.")
        else:
            df = pd.DataFrame(users, columns=["ID", "Username", "Email", "Failed Attempts", "Lock Until", "Is Admin"])
            for u in users:
                _ini = (u[1] or u[2])[:2].upper()
                _role = "Admin" if u[5] else "User"
                _rc = "#fca5a5" if u[5] else "#93c5fd"
                _locked_str = str(u[4]) if u[4] else ""
                _is_locked = _locked_str not in ["None", "null", "", "9999-12-31T23:59:59"]
                _stat = "Locked" if _is_locked else "Active"
                _sc = "#fbbf24" if _is_locked else "#34d399"
                st.markdown(
                    "<div style='background:#111827;border:1px solid #1f2937;"
                    "border-radius:12px;padding:12px 16px;margin-bottom:8px;"
                    "display:flex;align-items:center;gap:14px;'>"
                    "<div style='width:36px;height:36px;border-radius:50%;"
                    "background:linear-gradient(135deg,#ef4444,#f87171);"
                    "display:flex;align-items:center;justify-content:center;"
                    "font-weight:700;color:white;font-size:13px;flex-shrink:0;'>"
                    + _ini + "</div>"
                    "<div style='flex:1;min-width:0;'>"
                    "<div style='color:#f1f5f9;font-weight:600;font-size:13px;'>"
                    + (u[1] or "No username") + "</div>"
                    "<div style='color:#475569;font-size:11px;'>" + u[2] + "</div>"
                    "</div>"
                    "<div style='text-align:right;'>"
                    "<div style='font-size:11px;font-weight:600;color:" + _rc + ";'>" + _role + "</div>"
                    "<div style='font-size:11px;color:" + _sc + ";'>" + _stat + "</div>"
                    "</div></div>",
                    unsafe_allow_html=True
                )

            st.divider()
            col1, col2, col3, col4, col5 = st.columns(5)
            emails = [u[2] for u in users]

            with col1:
                st.markdown("**Unlock Account**")
                unlock_email = st.selectbox("Select User:", emails, key="unlock_sel")
                if st.button("Unlock"):
                    update_login_attempts(unlock_email, 0, None)
                    st.success(f"Unlocked {unlock_email}")

            with col2:
                st.markdown("**Lock Account**")
                lock_email = st.selectbox("Select User:", emails, key="lock_sel")
                if st.button("Lock", type="primary"):
                    if lock_email == os.getenv("ADMIN_EMAIL_ID"):
                        st.error("Cannot lock the Root Admin!")
                    else:
                        lock_user(lock_email)
                        st.success(f"Locked {lock_email}")

            with col3:
                st.markdown("**Promote to Admin**")
                promote_email = st.selectbox("Select User:", emails, key="prom_sel")
                if st.button("Promote"):
                    promote_user(promote_email)
                    st.success(f"{promote_email} is now an Admin")

            with col4:
                st.markdown("**Demote from Admin**")
                demote_email = st.selectbox("Select User:", emails, key="demote_sel")
                if st.button("Demote"):
                    if demote_email == os.getenv("ADMIN_EMAIL_ID"):
                        st.error("Cannot demote the Root Admin!")
                    else:
                        demote_user(demote_email)
                        st.success(f"{demote_email} is no longer an Admin")

            with col5:
                st.markdown("**Delete User**")
                del_email = st.selectbox("Select User:", emails, key="del_sel")
                if st.button("Delete Account", type="primary"):
                    if del_email == os.getenv("ADMIN_EMAIL_ID"):
                        st.error("Cannot delete the Root Admin!")
                    else:
                        delete_user(del_email)
                        st.success(f"Deleted user {del_email}")

    elif selected == "Activity Tracking":
        st.subheader("📜 Real-time System Activity")
        logs = get_all_activity_logs()
        if not logs:
            st.info("No system activity tracked yet.")
        else:
            df_logs = pd.DataFrame(logs, columns=["ID", "Email", "Section", "User Input", "AI Output", "Timestamp"])
            active_users = df_logs["Email"].nunique()
            st.metric("Total Unique Active Users", active_users)

            st.markdown("""
            <style>
            .timeline { border-left: 2px solid #334155; padding-left: 20px; margin-top: 20px; }
            .timeline-item { margin-bottom: 25px; position: relative; }
            .timeline-dot { width: 12px; height: 12px; background: #ef4444; border-radius: 50%; box-shadow: 0 0 10px rgba(239,68,68,0.6); position: absolute; left: -27px; top: 6px; }
            .timeline-card { background: rgba(30,41,59,0.55); border: 1px solid rgba(255,255,255,0.08); border-radius: 10px; padding: 16px; }
            .timeline-app { font-weight: 600; color: #fca5a5; font-size: 16px; }
            .timeline-time { font-size: 12px; color: #64748b; margin-top: 6px; }
            </style>
            <div class="timeline">
            """, unsafe_allow_html=True)
            for _, row in df_logs.iterrows():
                st.markdown(f'''
                <div class="timeline-item">
                    <div class="timeline-dot"></div>
                    <div class="timeline-card">
                        <div class="timeline-app">{row["Section"]} ({row["Email"]})</div>
                        <div style="margin-top:8px;color:#94a3b8;"><b>User:</b> {row["User Input"]}</div>
                ''', unsafe_allow_html=True)
                st.write(row["AI Output"])
                st.markdown(f'''
                        <div class="timeline-time">⏱ {row["Timestamp"]}</div>
                    </div>
                </div>
                ''', unsafe_allow_html=True)

    elif selected == "Data Visualization":
        st.subheader("📈 System Usage Analytics")
        logs = get_all_activity_logs()
        if not logs:
            st.info("Not enough data to visualize.")
        else:
            df_logs = pd.DataFrame(logs, columns=["ID", "Email", "Section", "Input", "Output", "Timestamp"])

            # Feature Popularity
            feature_counts = df_logs["Section"].value_counts().reset_index()
            feature_counts.columns = ["Feature", "Usage Count"]
            fig_feat = px.pie(feature_counts, values="Usage Count", names="Feature", title="Most Popular Features", hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
            st.plotly_chart(fig_feat, use_container_width=True)

            # Language Utilization — read from stored language column
            st.write("**Language Utilization**")
            lang_data = get_language_counts()
            if lang_data:
               df_lang = pd.DataFrame(lang_data, columns=["Language", "Count"])
               fig_lang = px.bar(df_lang, x="Language", y="Count", title="Language Distribution", text="Count", color="Language")
               st.plotly_chart(fig_lang, use_container_width=True)
            else:
               st.info("No language usage metrics recorded yet.")

            # Model Utilization estimation
            model_counts = {"Qwen 2.5": 0, "NLLB-200": 0, "spaCy NER": df_logs[df_logs["Section"] == "Knowledge Graph"].shape[0]}
            model_counts["Qwen 2.5"] = df_logs[df_logs["Section"].isin(["RAG Search", "Summarization"])].shape[0]
            model_counts["NLLB-200"] = df_lang["Count"].sum()
            df_mod = pd.DataFrame(list(model_counts.items()), columns=["Model", "Calls"])
            fig_mod = px.bar(df_mod, x="Model", y="Calls", title="AI Models Utilized", color_discrete_sequence=["#ef4444"])
            st.plotly_chart(fig_mod, use_container_width=True)

    elif selected == "Feedback Analysis":
        st.subheader("💬 User Feedback Sentiment Analysis")
        fb = get_all_feedback()
        if not fb:
            st.info("No feedback provided yet.")
        else:
            df_fb = pd.DataFrame(fb, columns=["Email", "Section", "Rating", "Comments", "Timestamp"])

            for _, row in df_fb.iterrows():
                st.markdown(f'''
                <div style="background: rgba(30,41,59,0.55); border: 1px solid rgba(255,255,255,0.08); border-radius: 10px; padding: 16px; margin-bottom: 12px;">
                    <div style="font-weight: 600; color: #38bdf8; font-size: 16px;">{row["Section"]} - ⭐ {row["Rating"]}/5</div>
                    <div style="color: #cbd5e1; margin-top: 8px;">"{row["Comments"]}"</div>
                    <div style="font-size: 12px; color: #64748b; margin-top: 8px;">👤 {row["Email"]}  ⏱ {row["Timestamp"]}</div>
                </div>
                ''', unsafe_allow_html=True)

            comments = " ".join([str(c) for c in df_fb["Comments"] if str(c).strip() and str(c) != "None"])
            if comments:
                from wordcloud import WordCloud
                import matplotlib.pyplot as plt
                st.write("**WordCloud of Feedback Comments**")
                wordcloud = WordCloud(width=800, height=400, background_color='#0b0f19', colormap='Reds').generate(comments)
                fig, ax = plt.subplots(figsize=(10,5))
                ax.imshow(wordcloud, interpolation='bilinear')
                ax.axis("off")
                fig.patch.set_facecolor('#0b0f19')
                st.pyplot(fig)
            else:
                st.info("No written comments to generate a WordCloud.")

    elif selected == "Data Export":
        st.subheader("📥 Export System Data (CSV)")

        # ── Users ──
        users = get_all_users()
        if users:
            df_users = pd.DataFrame(users, columns=["ID", "Username", "Email", "Failed Attempts", "Lock Until", "Is Admin"])
            st.markdown("**👥 Users**")
            for u in users:
                _i2 = (u[1] or u[2])[:2].upper()
                _r2 = "Admin" if u[5] else "User"
                _c2 = "#fca5a5" if u[5] else "#93c5fd"
                st.markdown(
                    "<div style='background:#111827;border:1px solid #1f2937;"
                    "border-radius:10px;padding:10px 14px;margin-bottom:6px;"
                    "display:flex;align-items:center;gap:12px;'>"
                    "<div style='width:32px;height:32px;border-radius:50%;"
                    "background:linear-gradient(135deg,#ef4444,#f87171);"
                    "display:flex;align-items:center;justify-content:center;"
                    "font-weight:700;color:white;font-size:12px;flex-shrink:0;'>"
                    + _i2 + "</div>"
                    "<div style='flex:1;'>"
                    "<div style='color:#f1f5f9;font-size:13px;font-weight:600;'>"
                    + (u[1] or "No username") + "</div>"
                    "<div style='color:#475569;font-size:11px;'>" + u[2] + "</div>"
                    "</div>"
                    "<div style='color:" + _c2 + ";font-size:12px;font-weight:600;'>" + _r2 + "</div>"
                    "</div>",
                    unsafe_allow_html=True
                )
            csv_users = df_users.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Users CSV", data=csv_users, file_name="users_export.csv", mime="text/csv")
        else:
            st.info("No users found.")

        st.divider()

        # ── Activity Logs ──
        logs = get_all_activity_logs()
        if logs:
            df_logs = pd.DataFrame(logs, columns=["ID", "Email", "Section", "Input", "Output", "Timestamp"])
            st.markdown("**📜 Activity Logs**")
            for log in logs[:50]:
                _ls = str(log[2]); _le = str(log[1]); _li = str(log[3]); _lt = str(log[5])
                st.markdown(
                    "<div style='background:#0f172a;border:1px solid #1e293b;"
                    "border-radius:10px;padding:10px 14px;margin-bottom:6px;'>"
                    "<div style='display:flex;justify-content:space-between;margin-bottom:4px;'>"
                    "<span style='color:#fca5a5;font-weight:600;font-size:12px;'>" + _ls + "</span>"
                    "<span style='color:#475569;font-size:11px;'>" + _lt + "</span>"
                    "</div>"
                    "<div style='color:#94a3b8;font-size:11px;margin-bottom:3px;'>&#128100; " + _le + "</div>"
                    "<div style='color:#64748b;font-size:11px;'><b>Input:</b> " + _li[:100] + ("..." if len(_li)>100 else "") + "</div>"
                    "</div>",
                    unsafe_allow_html=True
                )
            if len(logs) > 50:
                st.caption(f"Showing 50 of {len(logs)} records. Download CSV for all.")
            csv_logs = df_logs.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Activity Logs CSV", data=csv_logs, file_name="activity_logs.csv", mime="text/csv")
        else:
            st.info("No activity logs found.")

        st.divider()

        # ── Feedback ──
        fb = get_all_feedback()
        if fb:
            df_fb = pd.DataFrame(fb, columns=["Email", "Section", "Rating", "Comments", "Timestamp"])
            st.markdown("**💬 Feedback**")
            for row in fb:
                _fe = str(row[0]); _fse = str(row[1]); _fra = int(row[2]); _fts = str(row[4])
                _fco = str(row[3]) if str(row[3]) not in ["None",""] else "No comment"
                _stars = "★" * _fra + "☆" * (5 - _fra)
                st.markdown(
                    "<div style='background:#0f172a;border:1px solid #1e293b;"
                    "border-radius:10px;padding:10px 14px;margin-bottom:6px;'>"
                    "<div style='display:flex;justify-content:space-between;margin-bottom:4px;'>"
                    "<span style='color:#f1f5f9;font-size:12px;font-weight:600;'>&#128100; " + _fe + "</span>"
                    "<span style='color:#fbbf24;font-size:13px;'>" + _stars + "</span>"
                    "</div>"
                    "<div style='display:flex;justify-content:space-between;'>"
                    "<span style='color:#94a3b8;font-size:11px;'>" + _fse + "</span>"
                    "<span style='color:#475569;font-size:11px;'>" + _fts + "</span>"
                    "</div>"
                    "<div style='color:#64748b;font-size:11px;margin-top:4px;'>"" + _fco + ""</div>"
                    "</div>",
                    unsafe_allow_html=True
                )
            csv_fb = df_fb.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Feedback CSV", data=csv_fb, file_name="feedback_export.csv", mime="text/csv")
        else:
            st.info("No feedback found.")


# --------- forget page --------
def forgot_page():
    st.title("🔑 Forgot Password")
    render_header()

    email = st.text_input("Enter your registered email")

    method = st.radio(
        "Select Recovery Method",
        ["OTP Verification", "Security Question"]
    )

    if st.button("Continue"):

        user = get_user_by_email(email)

        if not user:
            st.toast("Email not registered", icon="❌")
            return

        st.session_state.pending_email = email

        # ⭐ Branch Logic
        if method == "OTP Verification":

            otp = generate_otp()
            send_otp_email(email, otp)

            st.session_state.otp = otp
            st.session_state.otp_time = datetime.utcnow()
            st.session_state.flow = "forgot_otp"

            go_to("OTP")

        else:
            st.session_state.flow = "forgot_security"
            go_to("SecurityQuestion")

    if st.button("⬅ Back"):
        go_to("Login")

#----- security questions ---------
def security_question_page():
    st.title("🔐 Security Verification")
    render_header()

    user = get_user_by_email(st.session_state.pending_email)

    st.write(user[4])  # security_question column

    answer = st.text_input("Your Answer")

    if st.button("Verify Answer"):

      if verify_text(answer, user[5]):

        if st.session_state.get("flow") == "forgot_security":
            go_to("SetNewPassword")
        else:
            st.toast("Unexpected flow", icon="⚠️")

      else:
        st.toast("Incorrect answer", icon="❌")

#--------- new password page ---------
def set_new_password_page():
    st.title("🔁 Set New Password")
    render_header()

    password = st.text_input("New Password", type="password")

    if st.button("Update Password"):

        valid, msg = validate_password(password)
        if not valid:
            st.toast(msg, icon="⚠️")
            return

        user = get_user_by_email(st.session_state.pending_email)

        # ✅ Load existing history
        history = json.loads(user[8] or "[]")

        # ✅ Prevent reuse of old passwords
        for old_hash in history:
            if verify_text(password, old_hash):
                st.toast("Cannot reuse old password", icon="⚠️")
                return

        new_hash = hash_text(password)

        # ✅ Update password
        update_password(st.session_state.pending_email, new_hash)

        # ✅ Update history properly
        history.insert(0, new_hash)
        history = history[:PASSWORD_HISTORY_COUNT]

        update_password_history(
            st.session_state.pending_email,
            json.dumps(history)
        )

        st.toast("Password reset successful", icon="✅")

        # Cleanup
        st.session_state.token = None
        go_to("Login")

# ---------- PROFILE TAB (Milestone 4) ----------
def profile_tab():
    render_sidebar()
    render_header()

    payload = verify_token(st.session_state.token)
    if not payload:
        st.toast('Session expired', icon='❌')
        go_to('Login')
        return

    profile_page.render_profile_page(payload['email'])

# ---------- ROUTER ----------
page = st.session_state.page

if page == "Signup":
    signup_page()
elif page == "Dashboard":
    dashboard_page()
elif page == "Reset":
    reset_page()
elif page == "Forgot":
    forgot_page()
elif page == "OTP":
    otp_page()
elif page == "SecurityQuestion":
    security_question_page()
elif page == "SetNewPassword":
    set_new_password_page()
elif page == "AdminDashboard":
    admin_dashboard()
elif page == "Readability":
    readability_page()
elif page == "RAG":
    rag_search_tab()
elif page == "Summarization":
    summarization_tab()
elif page == "Graph":
    graph_tab()
elif page == "History":
    overall_history_tab()
elif page == "Profile":  # Milestone 4
    profile_tab()
else:
    login_page()



Writing app.py


## Cell 12 — Set Secrets

In [ ]:
from google.colab import userdata
import os

os.environ['JWT_SECRET_KEY']     = userdata.get('JWT_SECRET_KEY')
os.environ['EMAIL_ID']           = userdata.get('EMAIL_ID')
os.environ['EMAIL_APP_PASSWORD'] = userdata.get('EMAIL_APP_PASSWORD')
os.environ['ADMIN_EMAIL_ID']     = userdata.get('ADMIN_EMAIL_ID')
os.environ['ADMIN_PASSWORD']     = userdata.get('ADMIN_PASSWORD')

# Verify
for var in ['JWT_SECRET_KEY','EMAIL_ID','EMAIL_APP_PASSWORD','ADMIN_EMAIL_ID','ADMIN_PASSWORD']:
    val = os.environ.get(var)
    print(f"{'✅' if val else '❌'} {var}")


✅ JWT_SECRET_KEY
✅ EMAIL_ID
✅ EMAIL_APP_PASSWORD
✅ ADMIN_EMAIL_ID
✅ ADMIN_PASSWORD


In [ ]:
#overwriting auth.py

with open("auth.py", "w") as f:
    f.write('''import os
import re
import bcrypt
import jwt
import random
import smtplib
from datetime import datetime, timedelta
from email.mime.text import MIMEText
from config import TOKEN_EXPIRY_MINUTES, JWT_ALGORITHM, EMAIL_ID, EMAIL_APP_PASSWORD

def hash_text(text):
    return bcrypt.hashpw(text.encode(), bcrypt.gensalt()).decode()

def verify_text(text, hashed):
    return bcrypt.checkpw(text.encode(), hashed.encode())

def validate_email(email):
    return re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$", email) is not None

def validate_password(password):
    if len(password) < 8:
        return False, "Password must be at least 8 characters"
    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least 1 uppercase letter"
    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least 1 lowercase letter"
    if not re.search(r"\\d", password):
        return False, "Password must contain at least 1 number"
    return True, "Valid password"

def generate_token(email):
    secret = os.getenv("JWT_SECRET_KEY")
    payload = {
        "email": email,
        "exp": datetime.utcnow() + timedelta(minutes=TOKEN_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, secret, algorithm=JWT_ALGORITHM)

def verify_token(token):
    try:
        secret = os.getenv("JWT_SECRET_KEY")
        return jwt.decode(token, secret, algorithms=[JWT_ALGORITHM])
    except:
        return None

def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp_email(receiver, otp):
    html_content = f"""
    <html><body style="font-family:Arial,sans-serif;background:#f4f4f4;padding:20px;">
        <div style="max-width:400px;margin:auto;background:white;padding:20px;
                    border-radius:10px;text-align:center;
                    box-shadow:0 4px 10px rgba(0,0,0,0.1);">
            <h2 style="color:#2c3e50;">PolicyNav Verification</h2>
            <p style="color:#555;">Use the OTP below to continue:</p>
            <div style="font-size:28px;font-weight:bold;letter-spacing:3px;
                        color:#00b894;margin:20px 0;">{otp}</div>
            <p style="color:#999;font-size:12px;">This OTP is valid for 10 minutes.</p>
        </div>
    </body></html>
    """
    msg = MIMEText(html_content, "html")
    msg["Subject"] = "PolicyNav OTP Verification"
    msg["From"] = EMAIL_ID
    msg["To"] = receiver
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_ID, EMAIL_APP_PASSWORD)
        server.send_message(msg)
''')
print("✅ auth.py rewritten. Now run Cell 14 to relaunch.")

✅ auth.py rewritten. Now run Cell 14 to relaunch.


In [ ]:
import os
secret = os.getenv("JWT_SECRET_KEY")
print(f"JWT_SECRET_KEY = {'✅ SET: ' + secret[:5] + '...' if secret else '❌ NOT SET'}")

JWT_SECRET_KEY = ✅ SET: super...


## Cell 13 — *(Optional)* Ingest PDFs

In [ ]:
import os, vector_store
docs_dir = os.path.join(APP_DIR, 'documents')
print('🔍 Scanning Google Drive for new PDFs...')
new_chunks = vector_store.ingest_documents(docs_dir)
if new_chunks > 0:
    print(f'✅ Ingested {new_chunks} new chunks into FAISS!')
else:
    print('⚡ Vector DB already up to date.')


🔍 Scanning Google Drive for new PDFs...
Parsing new document: PMAY_Urban_PMAY_1_crore_Flyer.pdf
Parsing new document: PMAY_Urban_PMAY Angikaar Flyer_29Aug_B.pdf
Parsing new document: PMAY_Urban_Innovative_20Technologies_20.pdf
Parsing new document: PMAY_Urban_TOT_Angikaar_final.pdf
Parsing new document: PMJDY_2Schemes-most-visible-PM-Yojanas.pdf
Parsing new document: PMJDY_hindi.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-November-2025.pdf
Parsing new document: PMJDY_dhan-accounts-claims-govt.pdf
Parsing new document: PMJDY_English.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-December-2025.pdf
Parsing new document: PMAY_Urban_ARH-EOI.pdf
⚡ Vector DB already up to date.


## Cell 14 — 🚀 Launch

In [ ]:
import os, subprocess, time
from google.colab import userdata
from pyngrok import ngrok

os.environ['APP_DIR'] = APP_DIR
ngrok_token = userdata.get('NGROK_AUTHTOKEN')

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    ngrok.kill()
    os.system('pkill -f streamlit')
    time.sleep(2)

    process = subprocess.Popen(
        ['streamlit', 'run', 'app.py'],
        env=os.environ.copy()
    )
    time.sleep(8)

    public_url = ngrok.connect(8501).public_url
    print(f'\n🚀 PolicyNav is live at: {public_url}\n')
    print('🛑 Press the Stop button or ENTER to shut down.')
    try: input('Press ENTER to stop...\n')
    except: pass
    finally: process.terminate(); ngrok.kill(); print('Stopped.')
else:
    print('❌ NGROK_AUTHTOKEN missing from Colab secrets.')



🚀 PolicyNav is live at: https://imprecatory-bradford-arboresque.ngrok-free.dev

🛑 Press the Stop button or ENTER to shut down.
